In [9]:
!pip install -q faster-whisper anthropic av

In [10]:
import os
import torch
from faster_whisper import WhisperModel
import anthropic
from google.colab import userdata

# Detect if CUDA (GPU) is available, otherwise fallback to CPU
if torch.cuda.is_available():
    device = "cuda"
    compute_type = "float16"
    print("Using GPU acceleration (cuda)")
else:
    device = "cpu"
    compute_type = "int8"
    print("CUDA GPU not available. Falling back to CPU (using int8 for efficiency)")

asr = WhisperModel("small", device=device, compute_type=compute_type)

# Safely retrieve the Anthropic API key
try:
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    # Fallback to manual environment variable or prompt if secret is missing
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        import getpass
        print("ANTHROPIC_API_KEY secret not found in Colab Secrets.")
        api_key = getpass.getpass("Please enter your Anthropic API Key manually: ")

client = anthropic.Anthropic(api_key=api_key)
print("Ready")

CUDA GPU not available. Falling back to CPU (using int8 for efficiency)
ANTHROPIC_API_KEY secret not found in Colab Secrets.
Please enter your Anthropic API Key manually: ··········
Ready


In [11]:
from faster_whisper import WhisperModel
from google.colab import userdata
import anthropic, time, torch, os

# Detect if CUDA (GPU) is available, otherwise fallback to CPU
if torch.cuda.is_available():
    device = "cuda"
    compute_type = "float16"
    print("Using GPU acceleration (cuda)")
else:
    device = "cpu"
    compute_type = "int8"
    print("CUDA GPU not available. Falling back to CPU (using int8 for efficiency)")

asr = WhisperModel("small", device=device, compute_type=compute_type)

# Safely retrieve the Anthropic API key with a fallback prompt
try:
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        import getpass
        print("ANTHROPIC_API_KEY secret not found in Colab Secrets.")
        api_key = getpass.getpass("Please enter your Anthropic API Key manually: ")

client = anthropic.Anthropic(api_key=api_key)
print("Ready")

CUDA GPU not available. Falling back to CPU (using int8 for efficiency)
ANTHROPIC_API_KEY secret not found in Colab Secrets.
Please enter your Anthropic API Key manually: ··········
Ready


In [14]:
import av

# Safely restore clean open function first to break any existing recursion loops
try:
    from av._api import open as clean_open
except ImportError:
    # Fallback if library structure differs
    import sys
    if 'av' in sys.modules:
        import importlib
        importlib.reload(av)
        clean_open = av.open

# Apply the bypass patch cleanly
def patched_open(*args, **kwargs):
    kwargs.pop('metadata_errors', None)
    return clean_open(*args, **kwargs)

av.open = patched_open
av._patched = True

# Path to your uploaded audio file
audio_file_path = "/content/Washroom q.m4a"

try:
    # Transcribe the audio file
    segments, info = asr.transcribe(audio_file_path, beam_size=5)

    print(f"Detected language '{info.language}' with probability {info.language_probability:.2f}")

    print("\nTranscription:")
    for segment in segments:
        print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")
except FileNotFoundError:
    print(f"Error: The file at '{audio_file_path}' was not found. Please upload your audio file or correct the path.")

Detected language 'hi' with probability 0.95

Transcription:
[0.00s -> 4.00s]  क्या में वाश्रूम के रास्ता जान सकती हूँ


In [16]:
# Define the transcription we received from the audio
hindi_transcription = "क्या में वाश्रूम के रास्ता जान सकती हूँ"

# Request a translation from Anthropic Claude (removing the temperature argument)
try:
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1000,
        system="You are an accurate translator. Translate the given text from Hindi to English.",
        messages=[
            {"role": "user", "content": f"Translate this Hindi text to English: {hindi_transcription}"}
        ]
    )
    translation = response.content[0].text
    print(f"Original Hindi: {hindi_transcription}")
    print(f"English Translation: {translation}")
except Exception as e:
    print(f"An error occurred during translation: {e}")

An error occurred during translation: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'Invalid bearer token'}, 'request_id': 'req_011CgbGiFcwQirnuK4cQpZR2'}


In [17]:
!pip install -q deep-translator

In [20]:
from deep_translator import GoogleTranslator

hindi_transcription = "क्या में वाश्रूम के रास्ता जान सकती हूँ"

try:
    # Translate from Hindi (hi) to English (en) without needing an API key
    translation = GoogleTranslator(source='hi', target='en').translate(hindi_transcription)
    print(f"Original Hindi: {hindi_transcription}")
    print(f"English Translation: {translation}")
except Exception as e:
    print(f"Translation error: {e}")

Translation error: Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function


In [21]:
!pip install -q translate

In [22]:
from translate import Translator

hindi_transcription = "क्या में वाश्रूम के रास्ता जान सकती हूँ"

try:
    # Set up translator from Hindi (hi) to English (en)
    translator = Translator(from_lang="hi", to_lang="en")
    translation = translator.translate(hindi_transcription)
    print(f"Original Hindi: {hindi_transcription}")
    print(f"English Translation: {translation}")
except Exception as e:
    print(f"Translation error: {e}")

Original Hindi: क्या में वाश्रूम के रास्ता जान सकती हूँ
English Translation: Can I find the way to the washroom


In [23]:
!pip install -q transformers sentencepiece

In [24]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

hindi_transcription = "क्या में वाश्रूम के रास्ता जान सकती हूँ"

try:
    print("Loading Meta M2M100 model and tokenizer...")
    model_name = "facebook/m2m100_418M"
    tokenizer = M2M100Tokenizer.from_pretrained(model_name)
    model = M2M100ForConditionalGeneration.from_pretrained(model_name)

    # Force source language to Hindi
    tokenizer.src_lang = "hi"
    encoded_hi = tokenizer(hindi_transcription, return_tensors="pt")

    # Generate translation forcing English target language token
    generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("en"))
    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    print(f"\nOriginal Hindi: {hindi_transcription}")
    print(f"M2M100 English Translation: {translation}")
except Exception as e:
    print(f"An error occurred with M2M translation: {e}")

Loading Meta M2M100 model and tokenizer...


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.71M [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 2.42MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.94GB            

model.safetensors: downloading bytes:           |  0.00B            


Original Hindi: क्या में वाश्रूम के रास्ता जान सकती हूँ
M2M100 English Translation: How can I know the way to the way?
